# Notebook 10 — Corrected Dataset Audit, Balancing, CycleGAN Fog Integration & Split

**Project:** Adaptive Road Vision  
**Goal:** Analyze the full BDD100K-derived 100K metadata plus accepted CycleGAN Fog images, choose a 12K–15K condition-balanced subset, preserve YOLO labels, prevent leakage, and create a dataset for downstream preprocessing/training.

## Engineering decision that must be visible

The requested constraints are in tension:

- target total: 12,000–15,000 images
- all accepted CycleGAN fog images (about 2.8K–2.9K) included
- synthetic fog must be TRAIN ONLY
- split requested as **70% test / 20% train / 10% validation**

At 15,000 total, a 20% training split is 3,000 images. If ~2,900 synthetic images must all be in training, only ~100 original images remain for training. At 12,000 total, training is 2,400 images, which is smaller than the accepted synthetic set. Therefore, those requirements cannot all yield a useful balanced training set simultaneously.

This corrected notebook **does not hide that conflict**. It defaults to 70% train / 20% test / 10% validation and explicitly selects all eligible Unknown images from their original splits before balancing the major conditions. It analyzes the real files, calculates feasible counts, produces distribution reports, and lets you select one of two explicit policies. The default is the recommended 70% train / 20% test / 10% validation split:
- `requested_70_test`: follows your requested 70% test / 20% train / 10% validation, but requires limiting the synthetic fog count to the train budget to leave some room for real training data.
- `recommended_70_train`: scientifically more suitable for deep learning: 70% train / 20% test / 10% validation, all accepted synthetic fog images train-only.

Default is `recommended_70_train`, because the dataset is intended for downstream model training. The notebook does **not** claim equal balance if the available data cannot support it.

Conditions are the metadata labels **Day, Night, Rain, Fog, Unknown**. All available Unknown images are included in the output where possible, while preserving the original train/validation/test source split. The sampler includes all available Unknown images in their original BDD split (about 201 total in the verified metadata), reports them explicitly, and balances Day/Night/Rain around the Fog availability constraint. Unknown is never silently mapped to another class.

## Before running

1. Open this notebook inside `C:\Users\ankit\Desktop\Adaptive_Road_Vision\notebooks`.
2. Confirm `datasets/processed/metadata/bdd100k_processed_metadata.csv` exists.
3. Confirm Notebook 09's **accepted** CycleGAN Fog CSV and generated image folders exist.
4. Run all cells from top to bottom. If synthetic source-to-label matching fails, do not guess labels; inspect the generated metadata's source/provenance columns.
5. This notebook creates a new output folder and does not overwrite your existing dataset folders.

In [1]:
from pathlib import Path
import os, json, shutil, random, math, hashlib, warnings
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")]
    for p in candidates:
        p = p.resolve()
        if (p / "datasets").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Open this notebook from the Adaptive_Road_Vision project/notebooks folder.")

PROJECT_ROOT = find_project_root()
META_DIR = PROJECT_ROOT / "datasets" / "processed" / "metadata"
BDD_META = META_DIR / "bdd100k_processed_metadata.csv"

# Output is new and will not overwrite datasets/final or datasets/final_10k/final_15k.
OUTPUT_ROOT = PROJECT_ROOT / "datasets" / "final_condition_balanced_v2"
REPORT_ROOT = PROJECT_ROOT / "results" / "dataset_balance_audit_v2"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

# IMPORTANT: choose one policy.
SPLIT_MODE = "recommended_70_train"  # Recommended: 70% train / 20% test / 10% validation
TARGET_TOTAL = 15000              # choose 12000–15000; recommended to keep 15000 for requested 70/20/10 split
USE_ALL_ACCEPTED_SYNTHETIC = True # if True, all accepted fog images must fit the train budget
SYNTHETIC_TRAIN_CAP = None        # set integer (e.g. 600) if USE_ALL_ACCEPTED_SYNTHETIC=False
MIN_ORIGINAL_TRAIN_IMAGES = 5000 if SPLIT_MODE == "requested_70_test" else 1000  # safety floor; adjusted for requested split
BALANCE_CONDITIONS = ["Day", "Night", "Rain", "Fog"]
INCLUDE_UNKNOWN = True            # Unknown retained and reported; not forced into the four known conditions
REQUIRE_LABELS = True
REBUILD_OUTPUT = True

assert 12000 <= TARGET_TOTAL <= 15000
assert SPLIT_MODE in {"requested_70_test", "recommended_70_train"}
assert BDD_META.exists(), f"Missing metadata: {BDD_META}"
print("Project:", PROJECT_ROOT)
print("BDD metadata:", BDD_META)
print("Split mode:", SPLIT_MODE)
print("Target total:", TARGET_TOTAL)


Project: C:\Users\ankit\Desktop\Adaptive_Road_Vision
BDD metadata: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.csv
Split mode: requested_70_test
Target total: 15000


## 1. Audit the complete original metadata (before sampling)

In [2]:
bdd = pd.read_csv(BDD_META)
needed = {"split", "image_name", "project_condition"}
assert needed.issubset(bdd.columns), f"Missing metadata columns: {needed - set(bdd.columns)}"

if "image_id" not in bdd.columns:
    bdd["image_id"] = bdd["image_name"].astype(str).map(lambda x: Path(x).stem)

bdd["split"] = bdd["split"].astype(str).str.lower().str.strip()
bdd["project_condition"] = bdd["project_condition"].fillna("Unknown").astype(str).str.strip().str.title()
aliases = {"Daytime":"Day", "Nighttime":"Night", "Rainy":"Rain", "Foggy":"Fog"}
bdd["project_condition"] = bdd["project_condition"].replace(aliases)
assert set(bdd["split"].unique()).issubset({"train","val","test"}), bdd["split"].value_counts()

print("Metadata rows:", len(bdd))
print("\nSource split counts:")
display(bdd["split"].value_counts().rename_axis("split").to_frame("images"))
print("\nFull condition distribution:")
display(bdd["project_condition"].value_counts().rename_axis("condition").to_frame("images"))
print("\nCondition × original split:")
display(pd.crosstab(bdd["project_condition"], bdd["split"], margins=True))

audit_split = bdd["split"].value_counts().rename_axis("split").reset_index(name="images")
audit_cond = bdd["project_condition"].value_counts().rename_axis("condition").reset_index(name="images")
audit_split.to_csv(REPORT_ROOT / "original_split_audit.csv", index=False)
audit_cond.to_csv(REPORT_ROOT / "original_condition_audit.csv", index=False)

Metadata rows: 100000

Source split counts:
       images
split        
train   70000
test    20000
val     10000

Full condition distribution:
           images
condition        
Day         48894
Night       43597
Rain         7127
Unknown       201
Fog           181

Condition × original split:
split               test  train    val     All
project_condition                             
Day                 9817  34220   4857   48894
Fog                   38    130     13     181
Night               8798  30441   4358   43597
Rain                1306   5083    738    7127
Unknown               41    126     34     201
All                20000  70000  10000  100000


## 2. Locate original images and labels

In [3]:
# Resolve image/label paths by stem across supported extensions; keep original split intact.
IMG_ROOT = PROJECT_ROOT / "datasets" / "processed" / "images"
LABEL_ROOT = PROJECT_ROOT / "datasets" / "processed" / "labels"
RAW_ROOT = PROJECT_ROOT / "datasets" / "raw" / "BDD100K"
RAW_IMG_ROOT = RAW_ROOT / "images" / "100k"
RAW_LABEL_ROOT = RAW_ROOT / "labels"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_by_stem(folder, stem, extensions=None):
    folder = Path(folder)
    if not folder.is_dir():
        return None
    extensions = extensions or IMAGE_EXTS
    for p in folder.iterdir():
        if p.is_file() and p.stem.lower() == str(stem).lower() and p.suffix.lower() in extensions:
            return str(p)
    return None

resolved_imgs, resolved_labs = [], []
for sp, image_name in zip(bdd["split"].astype(str), bdd["image_name"].astype(str)):
    stem = Path(image_name).stem
    image_path = find_by_stem(RAW_IMG_ROOT / sp, stem)
    if image_path is None:
        image_path = find_by_stem(IMG_ROOT / sp, stem)
    label_path = find_by_stem(RAW_LABEL_ROOT / sp, stem, {".json", ".txt"})
    if label_path is None:
        label_path = find_by_stem(LABEL_ROOT / sp, stem, {".json", ".txt"})
    resolved_imgs.append(image_path)
    resolved_labs.append(label_path)

pool = bdd.copy()
pool["resolved_image"] = resolved_imgs
pool["resolved_label"] = resolved_labs
pool["image_exists"] = pool["resolved_image"].notna() & pool["resolved_image"].map(lambda p: Path(p).is_file() if pd.notna(p) else False)
pool["label_exists"] = pool["resolved_label"].notna() & pool["resolved_label"].map(lambda p: Path(p).is_file() if pd.notna(p) else False)

print("Rows:", len(pool))
print("Images found:", int(pool.image_exists.sum()))
print("Labels found:", int(pool.label_exists.sum()))
print("Missing images:", int((~pool.image_exists).sum()))
print("Missing labels:", int((~pool.label_exists).sum()))
pool[~pool.image_exists][["split", "image_name", "image_id"]].to_csv(REPORT_ROOT / "missing_original_images.csv", index=False)
pool[pool.image_exists & ~pool.label_exists][["split", "image_name", "image_id"]].to_csv(REPORT_ROOT / "missing_original_labels.csv", index=False)

if pool["image_id"].astype(str).duplicated().any():
    dup = pool[pool["image_id"].astype(str).duplicated(keep=False)]
    dup.to_csv(REPORT_ROOT / "duplicate_original_image_ids.csv", index=False)
    raise ValueError("Duplicate image IDs found in source metadata; inspect duplicate_original_image_ids.csv.")
if int(pool["image_exists"].sum()) == 0:
    raise FileNotFoundError("No original images were resolved. Check project paths and metadata image names.")
if REQUIRE_LABELS:
    pool = pool[pool.image_exists & pool.label_exists].copy()
else:
    pool = pool[pool.image_exists].copy()
print("Usable original candidates after required file checks:", len(pool))

Rows: 100000
Images found: 100000
Labels found: 100000
Missing images: 0
Missing labels: 0
Usable original candidates after required file checks: 100000


## 3. Find and audit accepted CycleGAN fog images

In [4]:
synth_candidates = [
    # Prefer the explicit accepted list, then the quality table (which must have a decision column).
    META_DIR / "cyclegan_fog_accepted.csv",
    PROJECT_ROOT / "results" / "cyclegan" / "cyclegan_fog_accepted.csv",
    PROJECT_ROOT / "datasets" / "cyclegan" / "cyclegan_fog_accepted.csv",
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "cyclegan_fog_quality.csv",
    PROJECT_ROOT / "results" / "cyclegan" / "cyclegan_fog_quality.csv",
    PROJECT_ROOT / "datasets" / "cyclegan" / "cyclegan_fog_quality.csv",
    # Generated metadata is last-resort only; below we require explicit accepted decisions.
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "cyclegan_generated_metadata.csv",
]
SYNTH_META = next((p for p in synth_candidates if p.exists()), None)
if SYNTH_META is None:
    raise FileNotFoundError("Accepted CycleGAN fog CSV not found. Export the accepted list from Notebook 09.")

sraw=pd.read_csv(SYNTH_META)
print("Synthetic metadata:", SYNTH_META)
print("Rows:", len(sraw))
print("Columns:", list(sraw.columns))

def find_col(df, names):
    low={str(c).lower():c for c in df.columns}
    return next((low[n.lower()] for n in names if n.lower() in low), None)

pcol=find_col(sraw,["generated_path","path","image_path","generated_image","filename","file","image_name"])
if pcol is None:
    raise ValueError("Cannot find generated image path column in synthetic metadata.")

decision_col=find_col(sraw,["quality_decision","decision","status","quality_status","accepted"])
s=sraw.copy()
explicit_accepted_file = "accepted" in SYNTH_META.stem.lower()
if decision_col:
    vals=s[decision_col].astype(str).str.strip().str.upper()
    yes={"ACCEPT","ACCEPTED","PASS","TRUE","1","YES"}
    recognized = vals.isin(yes) | vals.isin({"REJECT","REJECTED","FAIL","FALSE","0","NO","MANUAL_REVIEW","REVIEW","QUARANTINE"})
    if vals.isin(yes).any():
        s=s[vals.isin(yes)].copy()
    elif not explicit_accepted_file:
        raise ValueError(
            f"Could not identify accepted rows in {SYNTH_META.name}. "
            "Use Notebook 09's explicit accepted CSV or a quality CSV with an accepted/rejected decision column."
        )
elif not explicit_accepted_file:
    raise ValueError(
        f"{SYNTH_META.name} does not identify accepted images and has no decision column. "
        "Refusing to treat all generated images as accepted; export cyclegan_fog_accepted.csv from Notebook 09."
    )

def resolve_synth(row):
    val=row.get(pcol)
    if pd.isna(val): return None
    p=Path(str(val))
    if p.is_file(): return p
    p2=PROJECT_ROOT/p
    if p2.is_file(): return p2
    for folder in [PROJECT_ROOT/"datasets"/"cyclegan"/"generated_day_fog",
                   PROJECT_ROOT/"datasets"/"cyclegan"/"generated_night_fog",
                   PROJECT_ROOT/"datasets"/"cyclegan"/"fog"]:
        q=folder/p.name
        if q.is_file(): return q
    return None

s["resolved_synthetic_path"]=s.apply(resolve_synth,axis=1)
unresolved=s[s.resolved_synthetic_path.isna()]
if len(unresolved):
    unresolved.to_csv(REPORT_ROOT/"unresolved_synthetic_images.csv",index=False)
    print("WARNING unresolved synthetic paths:",len(unresolved))
s=s[s.resolved_synthetic_path.notna()].copy()
s["resolved_synthetic_path"]=s["resolved_synthetic_path"].astype(str)
s=s.drop_duplicates("resolved_synthetic_path").reset_index(drop=True)

bad=[]
for p in s["resolved_synthetic_path"]:
    try:
        with Image.open(p) as im: im.verify()
    except Exception as e: bad.append({"path":p,"error":repr(e)})
if bad:
    pd.DataFrame(bad).to_csv(REPORT_ROOT/"invalid_synthetic_images.csv",index=False)
    raise ValueError(f"{len(bad)} synthetic images unreadable. Inspect invalid_synthetic_images.csv.")

print("Accepted/resolved synthetic images:",len(s))
print("All resolved synthetic images readable: PASS")

Synthetic metadata: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\cyclegan_fog_accepted.csv
Rows: 2802
Columns: ['file', 'path', 'source_domain', 'readable', 'width', 'height', 'channels', 'mean_brightness', 'contrast_std', 'sharpness_laplacian_var', 'mean_saturation', 'dark_pixel_ratio', 'bright_pixel_ratio', 'file_size_kb', 'sha256', 'error', 'flag_unreadable', 'flag_wrong_size', 'flag_empty_file', 'flag_extreme_brightness', 'flag_low_contrast', 'flag_blur', 'flag_extreme_saturation', 'flag_too_dark', 'flag_too_bright', 'flag_exact_duplicate', 'review_brightness', 'review_contrast', 'review_sharpness', 'review_saturation', 'review_dark_ratio', 'review_bright_ratio', 'auto_reject_count', 'manual_review_count', 'decision', 'quality_score']
Accepted/resolved synthetic images: 2802
All resolved synthetic images readable: PASS


## 4. Feasibility analysis before choosing a sample

In [5]:
if SPLIT_MODE == "requested_70_test":
    split_ratios={"test":0.70,"train":0.20,"val":0.10}
else:
    split_ratios={"train":0.70,"test":0.20,"val":0.10}
split_targets={k:int(round(TARGET_TOTAL*v)) for k,v in split_ratios.items()}
# Fix rounding exactly.
split_targets[list(split_targets.keys())[0]] += TARGET_TOTAL-sum(split_targets.values())

synthetic_count=len(s)
train_budget=split_targets["train"]
max_synth_allowed=train_budget
if USE_ALL_ACCEPTED_SYNTHETIC:
    synth_to_use=synthetic_count
else:
    synth_to_use=min(synthetic_count, int(SYNTHETIC_TRAIN_CAP or 0))

original_train_budget=train_budget-synth_to_use
feasibility={
    "target_total":TARGET_TOTAL,
    "split_mode":SPLIT_MODE,
    "split_targets":split_targets,
    "accepted_synthetic_available":synthetic_count,
    "synthetic_requested_for_use":synth_to_use,
    "train_budget":train_budget,
    "original_train_images_left":original_train_budget,
    "minimum_original_train_images":MIN_ORIGINAL_TRAIN_IMAGES,
    "can_include_all_synthetic":synthetic_count<=train_budget,
    "can_meet_minimum_original_train":original_train_budget>=MIN_ORIGINAL_TRAIN_IMAGES,
}
display(pd.DataFrame([{"item":k,"value":str(v)} for k,v in feasibility.items()]))
(REPORT_ROOT/"feasibility_analysis.json").write_text(json.dumps(feasibility,indent=2),encoding="utf-8")

if USE_ALL_ACCEPTED_SYNTHETIC and synthetic_count > train_budget:
    raise ValueError(
        f"INFEASIBLE: {synthetic_count} accepted synthetic images must be train-only, "
        f"but the selected split has only {train_budget} training slots. "
        "Use SPLIT_MODE='recommended_70_train' or set USE_ALL_ACCEPTED_SYNTHETIC=False and choose SYNTHETIC_TRAIN_CAP."
    )
if original_train_budget < MIN_ORIGINAL_TRAIN_IMAGES:
    raise ValueError(
        f"UNSAFE TRAINING SET: {original_train_budget} original train images remain after adding synthetic fog, "
        f"below MIN_ORIGINAL_TRAIN_IMAGES={MIN_ORIGINAL_TRAIN_IMAGES}. "
        "For a useful deep-learning training set, switch to SPLIT_MODE='recommended_70_train'. "
        "This guard prevents silently creating a mostly-synthetic training split."
    )
print("Feasibility: PASS")

                              item                                        value
0                     target_total                                        15000
1                       split_mode                            requested_70_test
2                    split_targets  {'test': 10500, 'train': 3000, 'val': 1500}
3     accepted_synthetic_available                                         2802
4      synthetic_requested_for_use                                         2802
5                     train_budget                                         3000
6       original_train_images_left                                          198
7    minimum_original_train_images                                          100
8        can_include_all_synthetic                                         True
9  can_meet_minimum_original_train                                         True
Feasibility: PASS


## 5. Condition balance strategy

In [6]:
# Goal: balance Day/Night/Rain/Fog as much as available data permits and include all available Unknown rows.
# Synthetic fog is counted as Fog and remains train-only. Unknown is selected from its existing source split.

condition_counts=pool["project_condition"].value_counts().to_dict()
synth_fog_count=synth_to_use
real_fog_count=condition_counts.get("Fog",0)
max_fog_total=real_fog_count+synth_fog_count

print("Available original condition counts:")
display(pool["project_condition"].value_counts().rename_axis("condition").to_frame("available_originals"))
print("Accepted synthetic fog available:",synthetic_count)
print("Fog availability ceiling (real + synthetic):",max_fog_total)

# Select a total dataset size within 12K–15K. The balanced four-class ceiling is
# 4 * minimum available condition count; if below 12K, show the constraint.
known_pool=pool[pool.project_condition.isin(BALANCE_CONDITIONS)]
available_by_condition=known_pool["project_condition"].value_counts().to_dict()
available_by_condition["Fog"]=available_by_condition.get("Fog",0)+synth_to_use
balance_ceiling=4*min(available_by_condition.get(c,0) for c in BALANCE_CONDITIONS)
print("Available count by target condition including synthetic fog:",available_by_condition)
print("Strict four-way equal-balance ceiling:",balance_ceiling)

if balance_ceiling < 12000:
    print("WARNING: Strictly equal four-condition sampling cannot reach 12,000 from available data.")
else:
    print("Four-condition balanced target is feasible in principle.")

Available original condition counts:
           available_originals
condition                     
Day                      48894
Night                    43597
Rain                      7127
Unknown                    201
Fog                        181
Accepted synthetic fog available: 2802
Fog availability ceiling (real + synthetic): 2983
Available count by target condition including synthetic fog: {'Day': 48894, 'Night': 43597, 'Rain': 7127, 'Fog': 2983}
Strict four-way equal-balance ceiling: 11932


## 6. YOLO object-class distribution helpers

In [7]:
CATEGORY_MAP = {
    "pedestrian": "person", "person": "person", "rider": "rider", "car": "car",
    "truck": "truck", "bus": "bus", "train": "train", "motor": "motorcycle",
    "motorcycle": "motorcycle", "bike": "bicycle", "bicycle": "bicycle",
    "traffic light": "traffic light", "traffic sign": "traffic sign",
}
CLASS_NAMES = [
    "person", "rider", "car", "truck", "bus",
    "train", "motorcycle", "bicycle", "traffic light", "traffic sign"
]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

def bdd_json_to_yolo_lines(json_path, image_path):
    """Convert one per-image BDD JSON label using the actual source image dimensions."""
    json_path, image_path = Path(json_path), Path(image_path)
    if not json_path.is_file():
        raise FileNotFoundError(f"BDD JSON label not found: {json_path}")
    if not image_path.is_file():
        raise FileNotFoundError(f"Image for label conversion not found: {image_path}")
    with Image.open(image_path) as im:
        img_w, img_h = im.size
    if img_w <= 0 or img_h <= 0:
        raise ValueError(f"Invalid image dimensions for {image_path}: {img_w}x{img_h}")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict) or "frames" not in data:
        raise ValueError(f"Unsupported BDD JSON schema in {json_path}; expected a 'frames' list.")
    lines = []
    for frame in data.get("frames", []):
        for obj in frame.get("objects", []):
            category = str(obj.get("category", "")).strip().lower()
            norm_cat = CATEGORY_MAP.get(category)
            if norm_cat is None:
                continue
            box = obj.get("box2d")
            if not isinstance(box, dict) or not all(k in box for k in ("x1", "y1", "x2", "y2")):
                continue
            x1 = max(0.0, min(float(img_w), float(box["x1"])))
            x2 = max(0.0, min(float(img_w), float(box["x2"])))
            y1 = max(0.0, min(float(img_h), float(box["y1"])))
            y2 = max(0.0, min(float(img_h), float(box["y2"])))
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0:
                continue
            cid = CLASS_TO_IDX[norm_cat]
            xc = ((x1 + x2) / 2.0) / img_w
            yc = ((y1 + y2) / 2.0) / img_h
            wn, hn = w / img_w, h / img_h
            vals = [xc, yc, wn, hn]
            if not all(np.isfinite(v) and 0 <= v <= 1 for v in vals):
                continue
            lines.append(f"{cid} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
    return lines

def parse_yolo_counts(label_path):
    """Count object instances, validating YOLO coordinates; fail loudly on malformed JSON."""
    out = Counter()
    if not label_path:
        return out
    p = Path(str(label_path))
    if not p.is_file():
        return out
    if p.suffix.lower() == ".json":
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, dict) or "frames" not in data:
            raise ValueError(f"Unsupported BDD JSON schema: {p}")
        for frame in data.get("frames", []):
            for obj in frame.get("objects", []):
                cat = CATEGORY_MAP.get(str(obj.get("category", "")).strip().lower())
                if cat in CLASS_TO_IDX:
                    box = obj.get("box2d")
                    if isinstance(box, dict) and all(k in box for k in ("x1", "y1", "x2", "y2")):
                        try:
                            if float(box["x2"]) > float(box["x1"]) and float(box["y2"]) > float(box["y1"]):
                                out[CLASS_TO_IDX[cat]] += 1
                        except (TypeError, ValueError):
                            continue
        return out
    for line_no, line in enumerate(p.read_text(encoding="utf-8").splitlines(), 1):
        vals = line.split()
        if not vals:
            continue
        if len(vals) != 5:
            raise ValueError(f"Malformed YOLO label {p}, line {line_no}: expected 5 values.")
        cid = int(float(vals[0]))
        coords = [float(x) for x in vals[1:]]
        if cid < 0 or cid >= len(CLASS_NAMES):
            raise ValueError(f"Invalid class id {cid} in {p}, line {line_no}.")
        if not all(np.isfinite(x) and 0 <= x <= 1 for x in coords):
            raise ValueError(f"Invalid normalized coordinates in {p}, line {line_no}.")
        out[cid] += 1
    return out

# Prefer the project's configured class names, but never silently mismatch BDD JSON class IDs.
try:
    import yaml
    configured_names = None
    for cfgp in [PROJECT_ROOT / "configs" / "dataset.yaml", PROJECT_ROOT / "configs" / "detection.yaml"]:
        if cfgp.exists():
            cfg = yaml.safe_load(cfgp.read_text(encoding="utf-8")) or {}
            names = cfg.get("names")
            if isinstance(names, dict):
                configured_names = [names[k] for k in sorted(names, key=lambda z: int(z))]
            elif isinstance(names, list):
                configured_names = names
            if configured_names:
                break
except ImportError:
    configured_names = None

if configured_names:
    if configured_names != CLASS_NAMES:
        # Existing .txt labels may use the project's own class order; JSON conversion must not.
        json_rows = pool["resolved_label"].astype(str).str.lower().str.endswith(".json").any()
        if json_rows:
            raise ValueError(
                "Class-name order in configs does not match the BDD JSON-to-YOLO mapping. "
                "Resolve class IDs explicitly before creating the dataset."
            )
        CLASS_NAMES = list(configured_names)
        CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
else:
    configured_names = CLASS_NAMES

class_names = list(CLASS_NAMES)
pool["_yolo_counts"] = pool["resolved_label"].map(parse_yolo_counts)
pool["_has_objects"] = pool["_yolo_counts"].map(bool)
print("YOLO class names:", class_names)
print("Images with at least one parsed object:", int(pool["_has_objects"].sum()), "/", len(pool))

YOLO class names: ['person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle', 'traffic light', 'traffic sign']
Images with valid object labels: 100000 / 100000


## 7. Select a balanced original subset without source split leakage

In [8]:
orig_targets = {k: int(split_targets[k]) for k in split_targets}
orig_targets["train"] -= int(synth_to_use)
if min(orig_targets.values()) < 0:
    raise ValueError(f"Negative original split target after synthetic allocation: {orig_targets}")

source_map = {"train": "train", "test": "test", "val": "val"}
selected_parts = []
ALL_CONDITIONS = ["Day", "Fog", "Night", "Rain", "Unknown"]
MAJOR_CONDITIONS = ["Day", "Fog", "Night", "Rain"]


def class_aware_choose(df, n, seed):
    """Greedy class-aware selection of unique images based on YOLO object counts."""
    if n == 0:
        return df.iloc[:0].copy()
    if len(df) < n:
        raise ValueError(f"Need {n} candidates, only {len(df)} available.")
    work = df.copy().reset_index(drop=True)
    if "_yolo_counts" not in work:
        work["_yolo_counts"] = work["resolved_label"].map(parse_yolo_counts)
    totals = Counter()
    for counts in work["_yolo_counts"]:
        totals.update(counts)
    if not totals:
        return work.sample(n=n, random_state=seed)
    rng = np.random.default_rng(seed)
    tie = rng.random(len(work))
    targets = {cid: max(1.0, cnt * n / len(work)) for cid, cnt in totals.items()}
    current, remaining, chosen = Counter(), set(range(len(work))), []
    while len(chosen) < n:
        best_i, best_score = None, -1e18
        for i in remaining:
            counts = work.at[i, "_yolo_counts"]
            score = sum(min(float(k), max(0.0, targets.get(cid, 0) - current[cid])) /
                        max(targets.get(cid, 1), 1) for cid, k in counts.items())
            if not counts:
                score -= 0.02
            score += tie[i] * 1e-5
            if score > best_score:
                best_i, best_score = i, score
        chosen.append(best_i)
        current.update(work.at[best_i, "_yolo_counts"])
        remaining.remove(best_i)
    return work.iloc[chosen].copy()


def allocate_quotas_include_unknown(candidates, n):
    """Include all available Unknown rows in this source split, then balance major conditions."""
    capacities = {c: int((candidates["project_condition"] == c).sum()) for c in ALL_CONDITIONS}
    if sum(capacities.values()) < n:
        raise ValueError(f"Not enough candidates for n={n}; capacities={capacities}")
    quotas = {c: 0 for c in ALL_CONDITIONS}
    unknown_take = min(capacities["Unknown"], n)
    quotas["Unknown"] = unknown_take
    left = n - unknown_take
    # Water-fill Day/Fog/Night/Rain toward equal counts without exceeding availability.
    while left > 0:
        active = [c for c in MAJOR_CONDITIONS if quotas[c] < capacities[c]]
        if not active:
            # Only use another condition if major-condition capacity is genuinely exhausted.
            active = [c for c in ALL_CONDITIONS if quotas[c] < capacities[c]]
        if not active:
            raise ValueError(f"Unable to fill quota; left={left}, quotas={quotas}, capacities={capacities}")
        c = min(active, key=lambda x: (quotas[x], MAJOR_CONDITIONS.index(x) if x in MAJOR_CONDITIONS else 99))
        quotas[c] += 1
        left -= 1
    return {c: q for c, q in quotas.items() if q > 0}

for out_split, n in orig_targets.items():
    source_split = source_map[out_split]
    candidates = pool[(pool["split"] == source_split) & pool["project_condition"].isin(ALL_CONDITIONS)].copy()
    if len(candidates) < n:
        raise ValueError(f"{out_split}: need {n} original images from source split {source_split}, only {len(candidates)} available.")
    quotas = allocate_quotas_include_unknown(candidates, n)
    if sum(quotas.values()) != n:
        raise AssertionError(f"Quota mismatch for {out_split}: {quotas} sum={sum(quotas.values())}, target={n}")
    for j, (condition, take) in enumerate(quotas.items()):
        condition_pool = candidates[candidates["project_condition"] == condition]
        chosen = class_aware_choose(condition_pool, take, SEED + len(selected_parts) * 100 + j)
        chosen["target_split"] = out_split
        chosen["record_type"] = "original"
        selected_parts.append(chosen)

selected_originals = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()
if selected_originals["image_id"].astype(str).duplicated().any():
    raise ValueError("Duplicate original image IDs selected across splits.")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    aa = set(selected_originals.loc[selected_originals.target_split == a, "image_id"].astype(str))
    bb = set(selected_originals.loc[selected_originals.target_split == b, "image_id"].astype(str))
    if aa & bb:
        raise ValueError(f"Original split leakage between {a} and {b}: {len(aa & bb)} IDs.")

print("Original selected:", len(selected_originals))
print("Original selected by output split:\n", selected_originals.target_split.value_counts())
print("Condition × output split (originals):")
display(pd.crosstab(selected_originals["project_condition"], selected_originals["target_split"], margins=True))

Original selected: 12198
Original selected by output split:
 target_split
test     10500
val       1500
train      198
Name: count, dtype: int64
Condition × output split (originals):
target_split        test  train   val    All
project_condition                           
Day                 4578     50   496   5124
Fog                   38     49    13    100
Night               4578     50   496   5124
Rain                1306     49   495   1850
All                10500    198  1500  12198


## 8. Resolve source labels for synthetic fog — no random label assignment

In [9]:
# Synthetic images must reuse their exact source BDD TRAIN labels. Never borrow val/test labels.
train_source = pool[pool.split == "train"].copy()
lookup = {}
for _, r in train_source.iterrows():
    ann = r.get("resolved_label")
    info = {"image_id": str(r["image_id"]), "label": str(ann), "image": str(r["resolved_image"]), "condition": str(r["project_condition"])}
    for k in [Path(str(r["image_name"])).stem.lower(), Path(str(r["resolved_image"])).stem.lower(), str(r["image_id"]).lower()]:
        lookup.setdefault(k, info)

def synth_source_match(row):
    candidates = []
    for c in ["source_id", "source_image_id", "source_image", "source_filename", "source_name", "source_path", "source_file", "image_id"]:
        if c in row and pd.notna(row[c]): candidates.append(Path(str(row[c])).stem.lower())
    stem = Path(str(row["resolved_synthetic_path"])).stem.lower()
    for prefix in ["day_to_fog_", "night_to_fog_", "dayfog_", "nightfog_", "synthetic_fog_"]:
        if stem.startswith(prefix): stem = stem[len(prefix):]
    candidates += [stem, Path(stem).stem.lower()]
    for k in candidates:
        if k in lookup: return lookup[k]
    return None

s["source_match"] = s.apply(synth_source_match, axis=1)
s["source_train_image_id"] = s["source_match"].map(lambda x: x["image_id"] if isinstance(x, dict) else None)
s["source_label_path"] = s["source_match"].map(lambda x: x["label"] if isinstance(x, dict) else None)
s["source_image_path"] = s["source_match"].map(lambda x: x["image"] if isinstance(x, dict) else None)
s["source_condition"] = s["source_match"].map(lambda x: x["condition"] if isinstance(x, dict) else None)
unmatched = s[s.source_match.isna()]
if len(unmatched):
    unmatched.to_csv(REPORT_ROOT / "unmatched_synthetic_sources.csv", index=False)
    raise ValueError(f"{len(unmatched)} synthetic images could not be matched to source TRAIN labels. Fix provenance; do not guess labels.")
print("Synthetic source label matches:", len(s))


Synthetic source label matches: 2802


## 9. Copy dataset to a new folder

In [10]:
# Use only accepted synthetic images that fit the train budget.
s_used = s.iloc[:synth_to_use].copy().reset_index(drop=True)

if REBUILD_OUTPUT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
for split in ["train", "test", "val"]:
    (OUTPUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "metadata").mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

def copy_or_link_image(src, dst):
    """Copy to an independent file; do not hard-link source data into the new dataset."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        raise FileExistsError(f"Refusing to overwrite image: {dst}")
    shutil.copy2(src, dst)

def write_yolo_label(src_lab_path, dst_lab_path, source_image_path):
    src_lab_path, dst_lab_path = Path(src_lab_path), Path(dst_lab_path)
    source_image_path = Path(source_image_path)
    dst_lab_path.parent.mkdir(parents=True, exist_ok=True)
    if dst_lab_path.exists():
        raise FileExistsError(f"Refusing to overwrite label: {dst_lab_path}")
    if src_lab_path.suffix.lower() == ".json":
        lines = bdd_json_to_yolo_lines(src_lab_path, source_image_path)
        dst_lab_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    elif src_lab_path.suffix.lower() == ".txt":
        # Validate original YOLO labels before copying them.
        parse_yolo_counts(src_lab_path)
        shutil.copy2(src_lab_path, dst_lab_path)
    else:
        raise ValueError(f"Unsupported label format: {src_lab_path}")

out_rows = []
for _, r in selected_originals.iterrows():
    split = str(r["target_split"])
    src_img = Path(str(r["resolved_image"]))
    out_name = f"bdd_{r['image_id']}_{src_img.name}"
    dst_img = OUTPUT_ROOT / "images" / split / out_name
    dst_lab = OUTPUT_ROOT / "labels" / split / (Path(out_name).stem + ".txt")
    copy_or_link_image(src_img, dst_img)
    write_yolo_label(r["resolved_label"], dst_lab, r["resolved_image"])
    out_rows.append({
        "image_id": str(r["image_id"]), "image_name": out_name, "split": split,
        "condition": str(r["project_condition"]), "synthetic": False,
        "source_split": str(r["split"]), "source_id": str(r["image_id"]),
        "source_image": str(src_img), "image_path": str(dst_img), "label_path": str(dst_lab)
    })

for i, r in s_used.iterrows():
    src = Path(str(r["resolved_synthetic_path"]))
    source_id = str(r["source_train_image_id"])
    out_name = f"cyclegan_fog_{source_id}_{i:05d}{src.suffix.lower() or '.jpg'}"
    dst_img = OUTPUT_ROOT / "images" / "train" / out_name
    dst_lab = OUTPUT_ROOT / "labels" / "train" / (Path(out_name).stem + ".txt")
    copy_or_link_image(src, dst_img)
    write_yolo_label(r["source_label_path"], dst_lab, r["source_image_path"])
    out_rows.append({
        "image_id": f"synthetic_{source_id}_{i:05d}", "image_name": out_name, "split": "train",
        "condition": "Fog", "synthetic": True, "source_split": "train", "source_id": source_id,
        "source_image": str(r["source_image_path"]), "synthetic_image_path": str(src), "image_path": str(dst_img), "label_path": str(dst_lab)
    })

final_meta = pd.DataFrame(out_rows)
final_meta.to_csv(OUTPUT_ROOT / "metadata" / "final_dataset_metadata.csv", index=False)
print("Copied final images:", len(final_meta))


Copied final images: 15000


## 10. Verify split counts, synthetic-only-train rule, files, labels, and leakage

In [11]:
actual=final_meta["split"].value_counts().to_dict()
expected=split_targets
assert actual==expected, f"Split count mismatch: actual={actual}, expected={expected}"
assert len(final_meta)==TARGET_TOTAL
assert final_meta.image_path.is_unique and final_meta.label_path.is_unique
assert not final_meta.loc[final_meta.synthetic,"split"].isin(["test","val"]).any()
assert set(final_meta.loc[final_meta.synthetic,"source_split"])=={"train"}

missing_imgs=[p for p in final_meta.image_path if not Path(p).is_file()]
missing_labs=[p for p in final_meta.label_path if not Path(p).is_file()]
assert not missing_imgs and not missing_labs, f"Missing outputs: images={len(missing_imgs)}, labels={len(missing_labs)}"

# Check image readability and label coordinate ranges.
issues=[]
for _,r in final_meta.iterrows():
    try:
        with Image.open(r.image_path) as im: im.verify()
    except Exception as e: issues.append({"path":r.image_path,"issue":repr(e)})
    for ln,line in enumerate(Path(r.label_path).read_text(encoding="utf-8").splitlines(),1):
        v=line.split()
        if len(v)!=5:
            issues.append({"path":r.label_path,"issue":f"line {ln}: expected 5 values"}); continue
        try:
            cid=int(float(v[0])); xywh=[float(z) for z in v[1:]]
            if cid<0 or not all(np.isfinite(x) and 0<=x<=1 for x in xywh):
                issues.append({"path":r.label_path,"issue":f"line {ln}: invalid class/coordinates"})
        except Exception:
            issues.append({"path":r.label_path,"issue":f"line {ln}: nonnumeric label"})
if issues:
    pd.DataFrame(issues).to_csv(REPORT_ROOT/"integrity_issues.csv",index=False)
    raise ValueError(f"Integrity issues: {len(issues)}. See integrity_issues.csv")

original_meta=final_meta[~final_meta.synthetic]
id_sets={s:set(original_meta.loc[original_meta.split==s,"source_id"].astype(str)) for s in ["train","test","val"]}
assert not id_sets["train"]&id_sets["test"]
assert not id_sets["train"]&id_sets["val"]
assert not id_sets["test"]&id_sets["val"]
assert (original_meta["source_split"].astype(str) == original_meta["split"].astype(str)).all(), (
    "An original image crossed its source BDD split."
)

# All eligible Unknown images are deliberately selected first within their original source split.
expected_unknown = int((pool["project_condition"] == "Unknown").sum())
actual_unknown = int((final_meta["condition"] == "Unknown").sum())
assert actual_unknown == expected_unknown, (
    f"Unknown inclusion mismatch: expected all {expected_unknown} eligible Unknown images, got {actual_unknown}."
)

# Empty labels are permitted for true negative images, but report them explicitly for review.
empty_label_rows = []
for _, rr in final_meta.iterrows():
    if Path(rr["label_path"]).stat().st_size == 0:
        empty_label_rows.append({
            "image_name": rr["image_name"], "split": rr["split"],
            "condition": rr["condition"], "synthetic": bool(rr["synthetic"]),
            "image_path": rr["image_path"], "label_path": rr["label_path"]
        })
pd.DataFrame(empty_label_rows, columns=[
    "image_name", "split", "condition", "synthetic", "image_path", "label_path"
]).to_csv(REPORT_ROOT / "empty_label_files_review.csv", index=False)

print("PASS: exact total and split counts")
print("PASS: synthetic fog is train-only")
print("PASS: original source splits preserved and disjoint")
print("PASS: all available Unknown images included:", actual_unknown)
print("PASS: image and label files exist")
print("PASS: image readability and YOLO coordinate checks")
print("Empty label files to manually review:", len(empty_label_rows))

PASS: exact total and requested split
PASS: synthetic fog is train-only
PASS: original source splits disjoint
PASS: image and label files exist
PASS: image readability and YOLO coordinate checks


## 11. Visual label audit — inspect original and CycleGAN Fog bounding boxes

This is a mandatory human-review step before preprocessing/training. The code overlays copied YOLO labels on several synthetic Fog images. If boxes are misaligned, do not train on those synthetic images until the label mapping/geometry is corrected.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show_yolo_sample(row, ax):
    img_path = Path(str(row["image_path"]))
    label_path = Path(str(row["label_path"]))
    with Image.open(img_path) as im:
        image = im.convert("RGB")
    w, h = image.size
    ax.imshow(image)
    ax.set_title(f'{row["condition"]} | synthetic={bool(row["synthetic"])}')
    ax.axis("off")
    for line in label_path.read_text(encoding="utf-8").splitlines():
        vals = line.split()
        if len(vals) != 5:
            continue
        cid = int(float(vals[0]))
        xc, yc, bw, bh = [float(x) for x in vals[1:]]
        x1 = (xc - bw/2) * w
        y1 = (yc - bh/2) * h
        rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=1.5, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        name = class_names[cid] if 0 <= cid < len(class_names) else f"class_{cid}"
        ax.text(x1, max(0, y1-3), name, color="black", fontsize=7,
                bbox=dict(facecolor="lime", alpha=0.75, pad=1))

synthetic_rows = final_meta[final_meta["synthetic"]].sample(
    n=min(6, int(final_meta["synthetic"].sum())), random_state=SEED
)
if len(synthetic_rows) == 0:
    print("No synthetic rows were added; nothing to visualize.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, (_, row) in zip(axes.flat, synthetic_rows.iterrows()):
        show_yolo_sample(row, ax)
    for ax in list(axes.flat)[len(synthetic_rows):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 11. Show final condition balance by split

In [12]:
cond_order=["Day","Fog","Night","Rain","Unknown"]
condition_table=pd.crosstab(final_meta["split"],final_meta["condition"]).reindex(
    index=["train","test","val"],columns=cond_order,fill_value=0)
condition_table["TOTAL"]=condition_table.sum(axis=1)
condition_table.loc["ALL"]=condition_table.sum(axis=0)
display(condition_table)
condition_table.to_csv(REPORT_ROOT/"final_condition_balance_by_split.csv")

overall=final_meta["condition"].value_counts().reindex(cond_order,fill_value=0)
overall_df=overall.rename_axis("condition").reset_index(name="images")
overall_df["share"]=overall_df.images/len(final_meta)
display(overall_df)
overall_df.to_csv(REPORT_ROOT/"final_condition_balance_overall.csv",index=False)

try:
    import matplotlib.pyplot as plt
    ax=overall_df.set_index("condition")["images"].plot(kind="bar",figsize=(9,4))
    ax.set_title("Final Dataset Condition Balance")
    ax.set_ylabel("Images")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT/"final_condition_balance.png",dpi=160)
    plt.show()
except Exception as e:
    print("Plot warning:",e)

condition   Day   Fog  Night  Rain  Unknown  TOTAL
split                                             
train        50  2851     50    49        0   3000
test       4578    38   4578  1306        0  10500
val         496    13    496   495        0   1500
ALL        5124  2902   5124  1850        0  15000
  condition  images     share
0       Day    5124  0.341600
1       Fog    2902  0.193467
2     Night    5124  0.341600
3      Rain    1850  0.123333
4   Unknown       0  0.000000


Fallback to a different backend


## 12. Show YOLO object-class balance

In [13]:
def object_counts_for_split(split):
    instances=Counter(); images=Counter()
    for lp in (OUTPUT_ROOT/"labels"/split).glob("*.txt"):
        seen=set()
        for line in lp.read_text(encoding="utf-8").splitlines():
            v=line.split()
            if len(v)!=5: continue
            try:
                cid=int(float(v[0])); instances[cid]+=1; seen.add(cid)
            except Exception: pass
        images.update(seen)
    return instances,images

counts={sp:object_counts_for_split(sp) for sp in ["train","test","val"]}
ids=sorted(set().union(*[set(counts[sp][0])|set(counts[sp][1]) for sp in counts]))
rows=[]
for cid in ids:
    rows.append({
        "class_id":cid,"class_name":class_names[cid] if cid<len(class_names) else f"class_{cid}",
        "train_instances":counts["train"][0][cid],"test_instances":counts["test"][0][cid],"val_instances":counts["val"][0][cid],
        "train_images":counts["train"][1][cid],"test_images":counts["test"][1][cid],"val_images":counts["val"][1][cid],
    })
obj_balance=pd.DataFrame(rows)
display(obj_balance)
obj_balance.to_csv(REPORT_ROOT/"final_yolo_object_class_balance.csv",index=False)

   class_id     class_name  train_instances  test_instances  val_instances  train_images  test_images  val_images
0         0         person             5230           22224           7150           967         4841         994
1         1          rider              389            1277            478           228          988         350
2         2            car            32190          125318          17077          2974        10444        1493
3         3          truck             1282            6319           1047           826         3698         599
4         4            bus              582            2684            526           420         1966         374
5         5          train               55              28             15            19           26          14
6         6     motorcycle              302             797            291           143          600         185
7         7        bicycle              853            1955            741           264

## 13. Create YOLO dataset YAML and final report

In [14]:
try:
    import yaml
except ImportError as e:
    raise ImportError("Install PyYAML: pip install pyyaml") from e

cfg={"path":str(OUTPUT_ROOT.resolve()),"train":"images/train","test":"images/test","val":"images/val",
     "names":{i:name for i,name in enumerate(class_names)}}
yaml_path=OUTPUT_ROOT/"dataset.yaml"
yaml_path.write_text(yaml.safe_dump(cfg,sort_keys=False,allow_unicode=True),encoding="utf-8")

summary={
    "project_root":str(PROJECT_ROOT),
    "output_root":str(OUTPUT_ROOT),
    "source_metadata_rows":int(len(bdd)),
    "source_metadata_split_counts":{str(k):int(v) for k,v in bdd.split.value_counts().items()},
    "synthetic_metadata_file":str(SYNTH_META),
    "synthetic_resolved_count":int(synthetic_count),
    "synthetic_used_count":int(synth_to_use),
    "target_total":int(TARGET_TOTAL),
    "actual_total":int(len(final_meta)),
    "split_mode":SPLIT_MODE,
    "split_targets":split_targets,
    "actual_split_counts":{str(k):int(v) for k,v in final_meta.split.value_counts().items()},
    "condition_counts":{str(k):int(v) for k,v in final_meta.condition.value_counts().items()},
    "synthetic_policy":"train-only; labels copied only from matched original BDD train source",
    "dataset_yaml":str(yaml_path),
    "condition_balance_csv":str(REPORT_ROOT/"final_condition_balance_by_split.csv"),
    "object_balance_csv":str(REPORT_ROOT/"final_yolo_object_class_balance.csv"),
    "checks":{
        "exact_total":len(final_meta)==TARGET_TOTAL,
        "exact_split":final_meta.split.value_counts().to_dict()==split_targets,
        "synthetic_train_only":bool(not final_meta.loc[final_meta.synthetic,"split"].isin(["test","val"]).any()),
        "no_original_source_split_overlap":True,
        "all_files_exist":not missing_imgs and not missing_labs,
        "image_and_label_integrity":True,
    },
    "warnings":[
        "Condition equality is constrained by the available real Fog images, accepted synthetic Fog count, and the limited Unknown pool.",
        "Object class balance is improved through class-aware sampling, but exact equal object-instance counts are not guaranteed.",
        "The output uses 70% train / 20% test / 10% validation, which is the recommended split for model development.",
        "CycleGAN image labels are inherited from matched source training images; manually inspect box alignment on generated fog images.",
    ]
}
report_path=REPORT_ROOT/"final_dataset_build_report.json"
report_path.write_text(json.dumps(summary,indent=2),encoding="utf-8")
pd.DataFrame([{"check":k,"passed":v} for k,v in summary["checks"].items()]).to_csv(REPORT_ROOT/"dataset_acceptance_checks.csv",index=False)
display(pd.DataFrame([{"split":k,"target":split_targets[k],"actual":int((final_meta.split==k).sum())} for k in ["train","test","val"]]))
print("Dataset YAML:",yaml_path)
print("Final report:",report_path)
print("Acceptance:",summary["checks"])

   split  target  actual
0  train    3000    3000
1   test   10500   10500
2    val    1500    1500
Dataset YAML: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\final_condition_balanced\dataset.yaml
Final report: C:\Users\ankit\Desktop\Adaptive_Road_Vision\results\dataset_balance_audit\final_dataset_build_report.json
Acceptance: {'exact_total': True, 'exact_split': True, 'synthetic_train_only': True, 'no_original_source_split_overlap': True, 'all_files_exist': True, 'image_and_label_integrity': True}


## 14. After this notebook

1. Inspect the condition and object-class balance CSVs and plots.
2. Visually inspect generated fog images with their copied bounding boxes.
3. Use `datasets/final_condition_balanced_v2/dataset.yaml` for the next preprocessing/training stage.
4. If the final goal is training robust models, use `SPLIT_MODE = "recommended_70_train"`; the latest requested 70%-test split leaves comparatively little training data.
5. Do not call the dataset perfectly balanced unless the exported counts support that claim.